# Phân Tích Hồ Sơ Dữ Liệu & Thống Kê Mô Tả — Lưới Điện DAEWOO

## Data Provenance (Nguồn gốc & Vòng đời dữ liệu)

| Mục | Chi tiết |
|---|---|
| **Tên Dataset** | Steel Industry Energy Consumption Dataset |
| **Người thực hiện** | Hoàng Nguyễn Minh Khang |
| **Ngày thực hiện** | 14/04/2026 (Cập nhật lần cuối) |
| **Nguồn gốc dữ liệu** | Công ty TNHH Thép DAEWOO (Gwangyang, Hàn Quốc) — [UCI ML Repository](https://archive.ics.uci.edu/dataset/851/steel+industry+energy+consumption) |
| **Phương thức thu thập** | Thông qua Website nghiên cứu của doanh nghiệp thép |

---

Notebook này thực hiện quy trình **Thống kê Mô tả** theo tiêu chuẩn tiền xử lý dữ liệu:
1. Mô tả Dataset (Data Profiling) & Từ điển dữ liệu
2. Thống kê mô tả (Descriptive Statistics) — Mean, Median, Std, IQR
3. Phân phối Chuẩn vs Phi Chuẩn (Normal vs Non-Normal)
4. Phát hiện Outlier (IQR method)
5. Tương quan & Missing Values
6. Tổng kết các bước Tiền xử lý

In [ ]:
"""
Data profiling and exploratory data analysis.

This script handles data loading, profiling, descriptive statistics,
outlier detection, and correlation analysis for the dataset.
"""
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Import từ src/
from src.data_loader import load_steel_data, inspect_data

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# ── Tải dữ liệu ──
df = load_steel_data()
print(f'Đã tải dữ liệu: {df.shape[0]:,} dòng × {df.shape[1]} cột')

---
## I. Mô Tả Dataset (Data Profiling)

In [ ]:
# ── Thông tin cơ bản ──
n_rows, n_cols = df.shape
mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print('=' * 60)
print('     THÔNG TIN CƠ BẢN VỀ TẬP DỮ LIỆU')
print('=' * 60)
print(f'  Tên dataset  : Steel Industry Energy Consumption')
print(f'  Số dòng (N)  : {n_rows:,}')
print(f'  Số cột       : {n_cols}')
print(f'  Khối lượng   : {mem_mb:.2f} MB')
print(f'  Trùng lặp    : {df.duplicated().sum()} dòng')
print(f'  Missing total: {df.isnull().sum().sum()} giá trị')
print('=' * 60)
print()
print('Kiểu dữ liệu theo cột:')
print(df.dtypes.value_counts().to_string())

### Từ Điển Dữ Liệu (Data Dictionary)

Bảng mô tả chi tiết từng biến số trong tập dữ liệu:

In [ ]:
# ── Từ điển dữ liệu — mô tả ý nghĩa vật lý của mỗi cột ──
column_descriptions = {
    'date': {
        'Giải thích': 'Thời điểm ghi nhận (mỗi 15 phút)',
        'Đơn vị': 'yyyy-mm-dd HH:MM:SS',
    },
    'Usage_kWh': {
        'Giải thích': 'Công suất tác dụng tiêu thụ (P)',
        'Đơn vị': 'kWh',
    },
    'Lagging_Current_Reactive.Power_kVarh': {
        'Giải thích': 'Công suất phản kháng trễ pha (Q lagging)',
        'Đơn vị': 'kVarh',
    },
    'Leading_Current_Reactive_Power_kVarh': {
        'Giải thích': 'Công suất phản kháng sớm pha (Q leading)',
        'Đơn vị': 'kVarh',
    },
    'CO2(tCO2)': {
        'Giải thích': 'Lượng phát thải CO2 tương ứng',
        'Đơn vị': 'tCO2 (tấn CO2)',
    },
    'Lagging_Current_Power_Factor': {
        'Giải thích': 'Hệ số công suất trễ pha (PF lagging)',
        'Đơn vị': 'Tỉ lệ (0-100)',
    },
    'Leading_Current_Power_Factor': {
        'Giải thích': 'Hệ số công suất sớm pha (PF leading)',
        'Đơn vị': 'Tỉ lệ (0-100)',
    },
    'NSM': {
        'Giải thích': 'Số giây tính từ nửa đêm (Number of Seconds from Midnight)',
        'Đơn vị': 'Giây (0 – 86400)',
    },
    'WeekStatus': {
        'Giải thích': 'Ngày trong tuần hay cuối tuần',
        'Đơn vị': 'Weekday / Weekend',
    },
    'Day_of_week': {
        'Giải thích': 'Thứ trong tuần',
        'Đơn vị': 'Monday – Sunday',
    },
    'Load_Type': {
        'Giải thích': 'Phân loại mức tải thiết bị',
        'Đơn vị': 'Light_Load / Medium_Load / Maximum_Load',
    },
}

# ── Tự động tạo bảng từ DataFrame ──
dict_rows = []
for col in df.columns:
    desc = column_descriptions.get(col, {})
    if pd.api.types.is_numeric_dtype(df[col]):
        mien = f'{df[col].min():.4f} → {df[col].max():.4f}'
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        mien = f'{df[col].min()} → {df[col].max()}'
    else:
        unique_vals = df[col].unique()
        mien = ', '.join(str(v) for v in unique_vals[:5])
        if len(unique_vals) > 5:
            mien += f' ... ({len(unique_vals)} giá trị)'

    dict_rows.append({
        'Tên viết tắt': col,
        'Giải thích': desc.get('Giải thích', '—'),
        'Kiểu DL': str(df[col].dtype),
        'Chiều dài': f'{df[col].notna().sum():,}',
        'Miền giá trị': mien,
        'Đơn vị tính': desc.get('Đơn vị', '—'),
    })

df_dict = pd.DataFrame(dict_rows)
df_dict.style.set_caption('Từ Điển Dữ Liệu — Steel Industry Energy Consumption')

---
## II. Thống Kê Mô Tả (Descriptive Statistics)

Phân tích xu hướng trung tâm (Central Tendency) và độ phân tán (Dispersion) cho tất cả biến số.

In [ ]:
# ── Bảng thống kê mô tả tổng quan ──
desc = df.describe().T
desc['IQR'] = desc['75%'] - desc['25%']
desc['Median'] = df.select_dtypes(include=[np.number]).median()

# Sắp xếp cột cho dễ đọc
desc = desc[['count', 'mean', 'Median', 'std', 'min', '25%', '50%', '75%', 'max', 'IQR']]
desc.columns = ['N', 'Mean', 'Median', 'Std', 'Min', 'Q1 (25%)', 'Q2 (50%)', 'Q3 (75%)', 'Max', 'IQR']

desc.style \
    .format(precision=4) \
    .background_gradient(cmap='Blues', subset=['Mean', 'Std']) \
    .set_caption('Bảng Thống Kê Mô Tả — Tất cả biến số')

### So sánh Mean vs Median

| | Khi nào dùng Mean | Khi nào dùng Median |
|---|---|---|
| **Phân bố** | Đều / Đối xứng | Lệch (Skewed) |
| **Dữ liệu** | Sạch, không có outlier | Bẩn, có nhiều outlier |
| **Ví dụ** | Chiều cao sinh viên | Thu nhập hộ gia đình |

> **Chú ý**: Nếu |Mean − Median| lớn → phân bố lệch → nên dùng Median.

In [ ]:
# ── So sánh Mean vs Median + Skewness ──
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

comparison = pd.DataFrame({
    'Mean': df[numeric_cols].mean(),
    'Median': df[numeric_cols].median(),
    '|Mean - Median|': (df[numeric_cols].mean() - df[numeric_cols].median()).abs(),
    'Skewness': df[numeric_cols].skew(),
    'Kurtosis': df[numeric_cols].kurtosis(),
})

# Đánh dấu: nếu |skewness| > 0.5 → lệch → nên dùng Median
comparison['Khuyến nghị'] = comparison['Skewness'].abs().apply(
    lambda s: 'Dùng Median (lệch)' if s > 0.5 else 'Dùng Mean (đối xứng)'
)

comparison.style \
    .format(precision=4, subset=['Mean', 'Median', '|Mean - Median|', 'Skewness', 'Kurtosis']) \
    .applymap(
        lambda v: 'background-color: #ffe6e6' if 'Median' in str(v) else '',
        subset=['Khuyến nghị']
    ) \
    .set_caption('So sánh Mean vs Median — Quyết định thước đo trung tâm')

In [ ]:
# ── Biểu đồ phân phối: Histogram + KDE, đánh dấu Mean & Median ──
n_numeric = len(numeric_cols)
n_rows_plot = (n_numeric + 2) // 3

fig, axes = plt.subplots(n_rows_plot, 3, figsize=(16, 4 * n_rows_plot))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    data = df[col].dropna()

    # Histogram + KDE
    ax.hist(data, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='white')
    data.plot.kde(ax=ax, color='navy', linewidth=1.5)

    # Đánh dấu Mean (đỏ, nét đứt) và Median (xanh lá, nét liền)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {data.mean():.2f}')
    ax.axvline(data.median(), color='green', linestyle='-', linewidth=1.5, label=f'Median = {data.median():.2f}')

    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylabel('')

# Ẩn các subplot trống
for idx in range(n_numeric, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Phân Phối Các Biến Số — Histogram + KDE (Mean vs Median)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## III. Phân Phối Chuẩn vs Phi Chuẩn

| Đặc điểm | Phân phối Chuẩn | Phân phối Phi Chuẩn |
|---|---|---|
| **Đồ thị** | Hàm chuông (Bell curve) | Lệch trái / Lệch phải |
| **Mean và Median** | Gần bằng nhau | Lệch nhau đáng kể |
| **Outlier** | Ít | Nhiều |

Kiểm định thống kê bằng **D'Agostino-Pearson test** (phù hợp N > 5000):

In [ ]:
# ── Kiểm định phân phối chuẩn (Normality Test) ──
normality_rows = []
for col in numeric_cols:
    data = df[col].dropna()
    skew = data.skew()
    kurt = data.kurtosis()

    # D'Agostino-Pearson test
    stat, p_value = stats.normaltest(data)

    normality_rows.append({
        'Biến số': col,
        'Skewness': round(skew, 4),
        'Kurtosis': round(kurt, 4),
        'Thống kê K²': round(stat, 2),
        'p-value': f'{p_value:.2e}',
        'Kết luận': 'Chuẩn (p ≥ 0.05)' if p_value >= 0.05 else 'Phi chuẩn (p < 0.05)',
    })

df_normality = pd.DataFrame(normality_rows).set_index('Biến số')
df_normality.style.set_caption('Kiểm Định Phân Phối Chuẩn — D\'Agostino-Pearson Test (α = 0.05)')

In [ ]:
# ── QQ-Plot cho các biến đại diện ──
# Chọn biến có skewness nhỏ nhất (gần chuẩn) và lớn nhất (lệch nhất)
skewness = df[numeric_cols].skew().abs()
most_normal = skewness.idxmin()
most_skewed = skewness.idxmax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label in zip(
    axes,
    [most_normal, most_skewed],
    ['Gần Chuẩn nhất', 'Lệch nhất']
):
    stats.probplot(df[col].dropna(), dist='norm', plot=ax)
    ax.set_title(f'QQ-Plot: {col}\n({label} — Skew = {df[col].skew():.3f})', fontsize=11)
    ax.get_lines()[0].set_markersize(2)

fig.suptitle('So Sánh QQ-Plot — Phân Phối Chuẩn vs Phi Chuẩn', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## IV. Phát Hiện Outlier (Ngoại Lai)

Sử dụng phương pháp **IQR (Interquartile Range)**:
- Giới hạn dưới = Q1 − 1.5 × IQR
- Giới hạn trên = Q3 + 1.5 × IQR
- Giá trị nằm ngoài khoảng [Giới hạn dưới, Giới hạn trên] → **Outlier**

In [ ]:
# ── Phát hiện Outlier theo phương pháp IQR ──
outlier_rows = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_outliers / len(df) * 100

    outlier_rows.append({
        'Biến số': col,
        'Q1': round(Q1, 4),
        'Q3': round(Q3, 4),
        'IQR': round(IQR, 4),
        'Giới hạn dưới': round(lower, 4),
        'Giới hạn trên': round(upper, 4),
        'Số Outlier': n_outliers,
        '% Outlier': f'{pct:.2f}%',
    })

df_outliers = pd.DataFrame(outlier_rows).set_index('Biến số')
df_outliers.style \
    .background_gradient(cmap='Reds', subset=['Số Outlier']) \
    .set_caption('Phát Hiện Outlier — Phương Pháp IQR (1.5×IQR)')

In [ ]:
# ── Box Plot cho tất cả biến số ──
fig, axes = plt.subplots(n_rows_plot, 3, figsize=(16, 4 * n_rows_plot))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    bp = ax.boxplot(
        df[col].dropna(), vert=True, patch_artist=True,
        boxprops=dict(facecolor='lightsteelblue', edgecolor='navy'),
        medianprops=dict(color='red', linewidth=2),
        flierprops=dict(marker='o', markerfacecolor='coral', markersize=3, alpha=0.5),
    )
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel(col)

for idx in range(n_numeric, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Box Plot — Phát Hiện Outlier Trực Quan', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## V. Tương Quan & Missing Values

In [ ]:
# ── Kiểm tra Missing Values ──
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

df_missing = pd.DataFrame({
    'Số Missing': missing,
    '% Missing': missing_pct,
})

print('═' * 50)
print('     KIỂM TRA GIÁ TRỊ THIẾU (MISSING VALUES)')
print('═' * 50)
if df.isnull().sum().sum() == 0:
    print('Không có giá trị thiếu — Dữ liệu KEPCO đảm bảo liền mạch.')
else:
    print(df_missing[df_missing['Số Missing'] > 0])

print(f'\nSố dòng trùng lặp: {df.duplicated().sum()}')

In [ ]:
# ── Ma trận tương quan (Correlation Matrix) ──
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Chỉ hiện nửa dưới
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.8},
    annot_kws={'size': 9},
)
plt.title('Ma Trận Tương Quan — Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# ── Tương quan mạnh (|r| > 0.7) ──
print('\nCác cặp biến có tương quan mạnh (|r| > 0.7):')
strong = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            strong.append({
                'Biến 1': corr.columns[i],
                'Biến 2': corr.columns[j],
                'r': round(r, 4),
                'Hướng': 'Thuận' if r > 0 else 'Nghịch',
            })
if strong:
    display(pd.DataFrame(strong))
else:
    print('  Không có cặp nào có |r| > 0.7')

---
## VI. Tổng Kết Các Bước Tiền Xử Lý

Theo quy trình tiền xử lý dữ liệu chuẩn, áp dụng cho tập **Steel Industry**:

| # | Bước | Trạng thái | Ghi chú |
|---|---|---|---|
| 1 | **Làm sạch DL** (duplicates, missing, outliers) | Đã kiểm tra | Không có missing values — dữ liệu thu thập tự động. Duplicates & outliers đã thống kê. |
| 2 | **Chuẩn hóa (Scaling)** | Feature Engineering | Áp dụng Z-Score / MinMaxScaler khi cần (xem `src/pipeline.py` bước 3-5). |
| 3 | **Mã hóa (Encoding)** | Feature Engineering | Load_Type, WeekStatus, Day_of_week → Label Encoding / One-Hot. |
| 4 | **Chọn lọc** (loại cột không cần) | Feature Selection | Dựa trên Correlation Matrix & Feature Importance (SHAP). |
| 5 | **Mất cân bằng (Class Imbalance)** | GAN Augmentation | Xem `src/gan_augmentation.py` — sinh mẫu bất thường nhân tạo. |
| 6 | **Chia dữ liệu** (Train/Test) | Modeling | Chia theo thời gian (time-based split), không shuffle. |
| 7 | **Xác thực** | Evaluation | F1-Score, AUC-ROC, Ablation Study. |